# 15 - Day 5 Assignment (Solution)

### Step 1 README: Environment Setup
This cell configures resilient imports and validates the local runtime context for end-to-end execution.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "edge_ai_course").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise ModuleNotFoundError(
        "Could not find src/edge_ai_course. Run from inside the 'day 5 sensor' folder or a child folder."
    )

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from edge_ai_course.signal_generator import SignalConfig, CLASS_ID_TO_NAME, generate_source_signals
from edge_ai_course.signal_dataset import build_sensor_dataset, save_dataset_artifacts, assert_no_source_leakage
from edge_ai_course.signal_features import extract_features, fft_components
from edge_ai_course.signal_models import train_classical_models, train_cnn_model, convert_to_tflite
from edge_ai_course.signal_visualization import plot_signal_and_fft, plot_feature_distribution, plot_confusion_matrix, plot_comparison_bars
from edge_ai_course.model_analysis import (
    build_comparison_table,
    write_day5_report,
    estimate_classical_pipeline_memory,
    estimate_cnn_pipeline_memory,
)

print("Environment ready for Notebook 15")
print(f"Project root: {PROJECT_ROOT}")
print(f"Pandas version: {pd.__version__}")


Environment ready for Notebook 15
Project root: C:\Users\thenebandaa\OneDrive - West Pharmaceutical Services, Inc\trainings\Edge AI\code\day 5 sensor
Pandas version: 2.2.2


### Step 2 README: End-to-End Pipeline Run
This cell runs data generation, classical model training, CNN training, and builds a combined comparison table.

In [ ]:
# End-to-end pipeline execution for final recommendation.
bundle = build_sensor_dataset(config=SignalConfig(), mode="quick", window_size=256, overlap_fraction=0.5)
assert_no_source_leakage(bundle)
save_dataset_artifacts(bundle)

features = bundle["feature_table"]
feature_columns = [c for c in features.columns if c not in {"split", "source_signal_id", "window_id", "class_id", "class_name"}]
train_df = features[features["split"] == "train"]
test_df = features[features["split"] == "test"]

classical_results = train_classical_models(
    train_df[feature_columns].to_numpy(dtype=np.float32),
    train_df["class_id"].to_numpy(dtype=np.int32),
    test_df[feature_columns].to_numpy(dtype=np.float32),
    test_df["class_id"].to_numpy(dtype=np.int32),
)

cnn_results = train_cnn_model(
    x_train=bundle["x_train"],
    y_train=bundle["y_train"],
    x_val=bundle["x_val"],
    y_val=bundle["y_val"],
    x_test=bundle["x_test"],
    y_test=bundle["y_test"],
    epochs=10,
    batch_size=64,
)

comparison = build_comparison_table(classical_results, cnn_results)
comparison

c:\Users\thenebandaa\OneDrive - West Pharmaceutical Services, Inc\trainings\Edge AI\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


### Step 3 README: Comparison Plots and Final Report
This cell saves side-by-side performance plots and writes the final markdown recommendation report.

In [ ]:
plot_dir = PROJECT_ROOT / "reports" / "day5" / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)
plot_comparison_bars(comparison, save_dir=plot_dir)

dataset_summary = {
    "mode": "quick",
    "sampling_rate_hz": bundle["config"].sampling_rate,
    "duration_seconds": bundle["config"].duration_seconds,
    "window_size": bundle["window_size"],
    "overlap_fraction": bundle["overlap_fraction"],
}
report_path = write_day5_report(dataset_summary, classical_results, cnn_results, comparison)
print(f"Saved comparison plots to: {plot_dir}")
report_path

### Step 4 README: TinyML Resource Reasoning
This cell estimates classical vs CNN memory footprints and prints deployment-focused interpretation notes.

In [ ]:
# TinyML reasoning support: rough RAM estimates and recommendation evidence.
best_classical_name = max(classical_results, key=lambda k: classical_results[k]["f1_macro"])
best_classical = classical_results[best_classical_name]

classical_ram = estimate_classical_pipeline_memory(
    feature_count=len(feature_columns),
    model_size_bytes=best_classical["model_size_bytes"],
)
cnn_ram = estimate_cnn_pipeline_memory(
    model_size_bytes=cnn_results["model_size_bytes"],
    parameter_count=cnn_results["parameter_count"],
)

print("Best classical model:", best_classical_name)
print("Classical RAM estimate:", classical_ram)
print("CNN RAM estimate:", cnn_ram)
print("Power note: actual energy per inference must be measured on device with voltage/current logging.")